In [1]:
# prepare_dataset.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import os
from sklearn.preprocessing import LabelEncoder
import pickle

In [2]:
df = pd.read_csv("BDCBC7196_Hematology_Dataset.csv")
df.head()

,Gender,Age,Hb,RBC,WBC,PLATELETS,LYMP,MONO,HCT,MCV,MCH,MCHC,RDW,PDW,MPV,PCT,Diagnosis
0,0,45,12.1,4.25,12300,404000.0,29.0,4.6,36.2,85.2,28.4,33.4,14.0,13.6,10.2,0.410,Anemia of Chronic Disease
1,0,58,12.3,4.34,12000,392000.0,30.0,5.1,37.1,85.5,28.3,33.1,14.0,13.8,10.2,0.390,Anemia of Chronic Disease
2,0,49,12.6,4.35,11300,387000.0,23.5,7.0,38.2,87.9,28.9,32.9,14.1,14.9,10.7,0.410,Anemia of Chronic Disease
3,0,43,12.0,4.30,5000,298000.0,43.1,6.5,35.8,83.4,27.9,33.5,13.7,15.3,8.5,0.254,Anemia of Chronic Disease
4,0,29,11.4,4.36,8720,267000.0,31.1,5.9,35.1,80.4,26.1,32.5,14.0,15.6,8.3,0.222,Anemia of Chronic Disease


In [ ]:
print("Initial shape:", df.shape)
print("Columns:", list(df.columns))
print("Missing per column:\n", df.isnull().sum())
print("Dtypes:\n", df.dtypes)

Initial shape: (7196, 17)
Columns: ['Gender', 'Age', 'Hb', 'RBC', 'WBC', 'PLATELETS', 'LYMP', 'MONO', 'HCT', 'MCV', 'MCH', 'MCHC', 'RDW', 'PDW', 'MPV', 'PCT', 'Diagnosis']
Missing per column:
 Gender       0
Age          0
Hb           0
RBC          0
WBC          0
PLATELETS    0
LYMP         0
MONO         0
HCT          0
MCV          0
MCH          0
MCHC         0
RDW          0
PDW          0
MPV          0
PCT          0
Diagnosis    0
dtype: int64
Dtypes:
 Gender         int64
Age            int64
Hb           float64
RBC          float64
WBC            int64
PLATELETS    float64
LYMP         float64
MONO         float64
HCT          float64
MCV          float64
MCH          float64
MCHC         float64
RDW          float64
PDW          float64
MPV          float64
PCT          float64
Diagnosis     object
dtype: object


In [ ]:
df = df.rename(columns={ "Hb": "HGB", "PLATELETS": "PLT", "Diagnosis": "label" })
print("After rename columns:", list(df.columns))

After rename columns: ['Gender', 'Age', 'HGB', 'RBC', 'WBC', 'PLT', 'LYMP', 'MONO', 'HCT', 'MCV', 'MCH', 'MCHC', 'RDW', 'PDW', 'MPV', 'PCT', 'label']


In [ ]:
ranges = {
    "HGB": (5, 18),
    "RBC": (3, 7),
    "WBC": (3000, 30000),
    "PLT": (50000, 600000),
    "MCV": (55, 115),
    "MCH": (20, 34),
    "MCHC": (30, 38),
    "RDW": (11, 18),
    "MPV": (7, 12),
    "PCT": (0.1, 0.5)
}

In [6]:
# 7) تقرير أولي: كام قيمة برا الرينج لكل عمود
print("Out-of-range counts BEFORE clipping:")
for col, (low, high) in ranges.items():
    if col in df.columns:
        n_low = (df[col] < low).sum()
        n_high = (df[col] > high).sum()
        print(f"{col}: below {low} -> {n_low}, above {high} -> {n_high}")


Out-of-range counts BEFORE clipping:
HGB: below 5 -> 6, above 18 -> 513
RBC: below 3 -> 1160, above 7 -> 33
WBC: below 3000 -> 254, above 30000 -> 1063
PLT: below 50000 -> 114, above 600000 -> 625
MCV: below 55 -> 93, above 115 -> 171
MCH: below 20 -> 437, above 34 -> 394
MCHC: below 30 -> 1358, above 38 -> 69
RDW: below 11 -> 0, above 18 -> 496
MPV: below 7 -> 285, above 12 -> 429
PCT: below 0.1 -> 69, above 0.5 -> 2804


In [7]:
# 8) نعمل نسخة ونطبق clipping (عشان ما نمسح صفوف كثيرة)
df_clipped = df.copy()
for col, (low, high) in ranges.items():
    if col in df_clipped.columns:
        df_clipped[col] = df_clipped[col].clip(lower=low, upper=high)

In [8]:
# 9) Features و label
if 'label' not in df_clipped.columns:
    raise ValueError("No 'label' column found. Check Diagnosis column name in CSV.")
X = df_clipped.drop("label", axis=1)
y = df_clipped["label"]

In [9]:
# 10) تحويل أي non-numeric لو موجود (ونفحص NaNs الناتجة)
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    for c in non_numeric:
        X[c] = pd.to_numeric(X[c], errors='coerce')
    if X.isnull().sum().sum() > 0:
        # هنا بنسقط الصفوف اللي فيها قيم غير قابلة للتحويل
        mask_valid = X.notnull().all(axis=1)
        X = X[mask_valid].reset_index(drop=True)
        y = y[mask_valid].reset_index(drop=True)

print("Final X shape before split:", X.shape)    # أعلى قيمة طبيًا 60-100 ألف

Final X shape before split: (7196, 16)


In [10]:
# 11) Stratified split: Train 70% | Val 15% | Test 15%
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

print("Shapes after split:", X_train.shape, X_val.shape, X_test.shape)


Shapes after split: (5037, 16) (1079, 16) (1080, 16)


In [11]:
# 12) طباعة توزيع اللابلز للتأكد من الـ stratify
def print_label_dist(s, name):
    vc = s.value_counts()
    print(f"Label dist {name}:")
    print((vc / vc.sum()).round(4))
print_label_dist(y_train, "train")
print_label_dist(y_val, "val")
print_label_dist(y_test, "test")

Label dist train:
label
Anemia of Chronic Disease    0.1944
Normocytic Anemia            0.1757
Bacterial Infections         0.1612
Thrombocytosis               0.0864
Infectious Mononucleosis     0.0655
Beta-Thalassemia Minor       0.0600
Normal                       0.0578
Viral Infections             0.0449
Microcytic Anemia            0.0324
Leukopenia                   0.0324
Iron Deficiency              0.0320
Leukocytosis                 0.0238
Thrombocytopenia             0.0161
Chronic Leukemias            0.0137
Polycythemia Vera            0.0040
Name: count, dtype: float64
Label dist val:
label
Anemia of Chronic Disease    0.1946
Normocytic Anemia            0.1752
Bacterial Infections         0.1613
Thrombocytosis               0.0862
Infectious Mononucleosis     0.0658
Beta-Thalassemia Minor       0.0602
Normal                       0.0575
Viral Infections             0.0445
Leukopenia                   0.0324
Iron Deficiency              0.0315
Microcytic Anemia         

In [12]:
# 13) Scaling - StandardScaler: fit on train only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [13]:
# 14) تحويل لـ DataFrame و إعادة اضافة label
train_df = pd.DataFrame(X_train_scaled, columns=X.columns)
train_df['label'] = y_train.reset_index(drop=True)
val_df = pd.DataFrame(X_val_scaled, columns=X.columns)
val_df['label'] = y_val.reset_index(drop=True)
test_df = pd.DataFrame(X_test_scaled, columns=X.columns)
test_df['label'] = y_test.reset_index(drop=True)


In [15]:
# 15) حفظ CSVs و scaler
out_dir=r"C:\Users\ashra\Desktop\NN"
train_df.to_csv(os.path.join(out_dir, 'train_scaled.csv'), index=False)
val_df.to_csv(os.path.join(out_dir, 'val_scaled.csv'), index=False)
test_df.to_csv(os.path.join(out_dir, 'test_scaled.csv'), index=False)
joblib.dump(scaler, os.path.join(out_dir, 'standard_scaler.joblib'))

print("Processed files saved to", out_dir)

Processed files saved to C:\Users\ashra\Desktop\NN


In [2]:
# 1) Load original scaled data
# ============================
paths = {
    "train": "train_scaled.csv",
    "val": "val_scaled.csv",
    "test": "test_scaled.csv"
}

dfs = {k: pd.read_csv(v) for k, v in paths.items()}

In [3]:
# 2) Fit LabelEncoder on **train labels only**
# ============================================
le = LabelEncoder()
le.fit(dfs["train"]["label"])

LabelEncoder()

In [4]:
# 3) Create mapping (text → number)
# ============================================
mapping = {cls: int(le.transform([cls])[0]) for cls in le.classes_}
print("\n=== LABEL MAPPING ===")
for k, v in mapping.items():
    print(f"{v} → {k}")


=== LABEL MAPPING ===
0 → Anemia of Chronic Disease
1 → Bacterial Infections
2 → Beta-Thalassemia Minor
3 → Chronic Leukemias
4 → Infectious Mononucleosis
5 → Iron Deficiency
6 → Leukocytosis
7 → Leukopenia
8 → Microcytic Anemia
9 → Normal
10 → Normocytic Anemia
11 → Polycythemia Vera
12 → Thrombocytopenia
13 → Thrombocytosis
14 → Viral Infections


In [5]:
# 4) Transform labels in ALL splits using same encoder
# ===================================================
for k, df in dfs.items():
    df["label"] = le.transform(df["label"])
    df.to_csv(f"{k}_FINAL.csv", index=False)
    print(f"Saved {k}_FINAL.csv")

Saved train_FINAL.csv
Saved val_FINAL.csv
Saved test_FINAL.csv


In [6]:
# 5) Save LabelEncoder for inference (important)
# ============================================
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("\nlabel_encoder.pkl saved successfully.")


label_encoder.pkl saved successfully.


In [9]:
df['label'].value_counts().sort_values()

label
11      4
3      15
12     17
6      25
5      35
7      35
8      35
14     49
9      62
2      65
4      71
13     93
1     174
10    190
0     210
Name: count, dtype: int64

In [14]:
dfs["train"]["label"].value_counts()

label
0     979
10    885
1     812
13    435
4     330
2     302
9     291
14    226
8     163
7     163
5     161
6     120
12     81
3      69
11     20
Name: count, dtype: int64

In [15]:
dfs["val"]["label"].value_counts()

label
0     210
10    189
1     174
13     93
4      71
2      65
9      62
14     48
7      35
5      34
8      34
6      26
12     18
3      15
11      5
Name: count, dtype: int64

In [16]:
dfs["test"]["label"].value_counts()

label
0     210
10    190
1     174
13     93
4      71
2      65
9      62
14     49
5      35
7      35
8      35
6      25
12     17
3      15
11      4
Name: count, dtype: int64